# 1. Configuração IQ Option e login

In [1]:
!pip install -U --quiet https://github.com/iqoptionapi/iqoptionapi/archive/refs/heads/master.zip

     \ 17.2 MB 16.7 MB/s 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.6/200.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.8/521.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.6/274.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.3/92.3 kB 6.7 MB/s eta 0:00:00


In [2]:
from iqoptionapi.stable_api import IQ_Option

email = "sgtambrosio2019@gmail.com"
password = "YOUR_SECRET_HERE"
api = IQ_Option(email, password)
check, reason = api.connect()
print(check, reason)

True None


# 2. Configuração LangChain

In [3]:
!pip install -U --quiet langgraph langchain_anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.4/198.4 kB 9.6 MB/s eta 0:00:00


In [4]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["FINANCIAL_DATASETS_API_KEY"] = "OUR_FINANCIAL_DATASETS_API_KEY"

In [5]:
import json
import pandas as pd
import requests
from datetime import timedelta
import matplotlib.pyplot as plt

# Import your agent's dependencies
from langchain_anthropic.chat_models import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, MessagesState, StateGraph, START
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [6]:
llm = ChatAnthropic(model="claude-3-haiku-20240307")

# 3. Criar Agente de Dados de Mercado

In [7]:
def market_data_agent(state: MessagesState):
    """Responsável pela coleta e pré-processamento de dados de mercado"""
    messages = state["messages"]
    params = messages[-1].additional_kwargs

    # Obtenha os dados históricos de preços
    historical_data = get_price_data(params["ticker"])

    # Calcular os sinais de negociação
    signals = calculate_trading_signals(historical_data)

    # Crie a mensagem do agente de dados de mercado
    # usando os sinais e dados históricos
    message = HumanMessage(
        content=f"""
        Aqui estão os sinais de negociação para {params["ticker"]}:
        Preco atual: ${signals['current_price']:.2f}
        SMA 5: {signals['sma_5_curr']:.2f}
        SMA 5 Anterior: {signals['sma_5_prev']:.2f}
        SMA 20: {signals['sma_20_curr']:.2f}
        SMA 20 Anterior: {signals['sma_20_prev']:.2f}
        """,
        name="market_data_agent",
    )

    return {"messages": messages + [message]}

# 4. Criar Agente Quantitativo

In [8]:
def quant_agent(state: MessagesState):
    """Analisa indicadores técnicos e gera sinais de negociação"""
    last_message = state["messages"][-1]

    summary_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                Você é um analista técnico/quant de fundos de hedge.
                Você recebe sinais de negociação para uma ação.
                Analise os sinais e forneça uma recomendação.
                - sinal: otimista | pessimista | neutro,
                - confiança: <float entre 0 e 1>
                """
            ),
            MessagesPlaceholder(variable_name="messages"),
            (
                "human",
                f"""Com base nos sinais de negociação abaixo, analise os dados e forneça sua avaliação.

                Análise de negociação: {last_message.content}

                Inclua apenas seu sinal de negociação e confiança no resultado.
                """
            ),
        ]
    )

    chain = summary_prompt | llm

    result = chain.invoke(state).content
    message = HumanMessage(
        content=f"Aqui está a análise de negociação e minha recomendação:{result}",
        name="quant_agent",
    )

    return {"messages": state["messages"] + [message]}

# 5. Criar Agente de Gestão de Riscos

In [9]:
def risk_management_agent(state: MessagesState):
    """Avalia o risco do portfólio e define limites de posição"""
    portfolio = state["messages"][0].additional_kwargs["portfolio"]
    last_message = state["messages"][-1]

    risk_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                Você é um especialista em gerenciamento de risco.
                Seu trabalho é dar uma olhada na análise de negociação e
                avaliar a exposição do portfólio e recomendar o dimensionamento da posição.
                Forneça o seguinte em sua saída (não como um JSON):
                - max_position_size: <float maior que 0>,
                - risk_score: <int entre 1 e 10>
                """
            ),
            MessagesPlaceholder(variable_name="messages"),
            (
                "human",
                f"""Com base na análise de negociação abaixo, forneça sua avaliação de risco.

                Análise de negociação: {last_message.content}

                Aqui está o portfólio atual:
                Portfólio:
                Dinheiro: ${portfolio['cash']:.2f}
                Posição atual: {portfolio['stock']} ações

                Inclua apenas o tamanho máximo da posição e a pontuação de risco em sua saída.
                """
            ),
        ]
    )
    chain = risk_prompt | llm
    result = chain.invoke(state).content
    message = HumanMessage(
        content=f"Aqui está a recomendação de gerenciamento de risco: {result}",
        name="risk_management",
    )
    return {"messages": state["messages"] + [message]}

# 6. Criar Agente de Gerenciamento de Portifólio

In [ ]:
def portfolio_management_agent(state: MessagesState):
    """Toma decisões finais de negociação e gera ordens"""
    portfolio = state["messages"][0].additional_kwargs["portfolio"]
    last_message = state["messages"][-1]

    portfolio_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                Você é um gerente de portfólio que toma decisões finais de negociação.
                Seu trabalho é tomar uma decisão de negociação com base nos dados de gerenciamento de risco.
                Forneça o seguinte em sua saída:
                - "action": "call" | "put" | "hold",
                - "quantity": <int positivo>
                Compre somente se tiver dinheiro disponível.
                A quantidade que você compra deve ser menor ou igual ao tamanho máximo da posição.
                Venda somente se tiver ações no portfólio para vender.
                A quantidade que você vende deve ser menor ou igual à posição atual.
                """
            ),
            MessagesPlaceholder(variable_name="messages"),
            (
                "human",
                f"""
               Com base nos dados de gerenciamento de risco abaixo, tome sua decisão de negociação.

              Dados de gerenciamento de risco: {last_message.content}

              Aqui está o portfólio atual:
              Portfólio:
              Dinheiro: ${portfolio['cash']:.2f}
              Posição atual: {portfolio['stock']} ações

              Inclua apenas a ação e a quantidade.

              Lembre-se, a ação deve ser comprar, vender ou manter.
              Você só pode comprar se tiver dinheiro disponível.
              Você só pode vender se tiver ações no portfólio para vender.
                """
            ),
        ]
    )

    chain = portfolio_prompt | llm
    result = chain.invoke(state).content
    return {"messages": [HumanMessage(content=result, name="portfolio_management")]}

# 7. Criar Gráfico de Agente

In [ ]:
# Define the new workflow
workflow = StateGraph(MessagesState)

# Add nodes
workflow.add_node("market_data_agent", market_data_agent)
workflow.add_node("quant_agent", quant_agent)
workflow.add_node("risk_management_agent", risk_management_agent)
workflow.add_node("portfolio_management_agent", portfolio_management_agent)

# Define the workflow
workflow.add_edge(START, "market_data_agent")
workflow.add_edge("market_data_agent", "quant_agent")
workflow.add_edge("quant_agent", "risk_management_agent")
workflow.add_edge("risk_management_agent", "portfolio_management_agent")
workflow.add_edge("portfolio_management_agent", END)

app = workflow.compile()

In [ ]:
# Atualize a função run_agent para incluir o estado do portfólio
def run_agent(ticker: str, end_date: str, portfolio: dict):
    final_state = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Tome uma decisão de negociação com base nos dados fornecidos.",
                    additional_kwargs={
                        "ticker": ticker,
                        "end_date": end_date,
                    },
                )
            ]
        },
        config={"configurable": {"thread_id": 42}},
    )
    return final_state["messages"][-1].content

# 8. Obter o preço das ações e os sinais de negociação

In [ ]:
# Defina uma função para calcular sinais de negociação
def calculate_trading_signals(historical_data: pd.DataFrame) -> dict:
    """Calcular sinais de negociação com base na estratégia de crossover SMA"""
    # Calcular SMAs
    sma_5 = historical_data["close"].rolling(window=5).mean()
    sma_20 = historical_data["close"].rolling(window=20).mean()

    # Obtenha os dois últimos pontos de cada SMA para verificar se há cruzamento
    sma_5_prev, sma_5_curr = sma_5.iloc[-2:]
    sma_20_prev, sma_20_curr = sma_20.iloc[-2:]

    return {
        "current_price": historical_data["close"].iloc[-1],
        "sma_5_curr": sma_5_curr,
        "sma_5_prev": sma_5_prev,
        "sma_20_curr": sma_20_curr,
        "sma_20_prev": sma_20_prev,
    }

In [ ]:
# Mova a função de busca de dados de preço para o escopo do agente
def get_price_data(ativo,
                   quantidade_total=100,
                   unidade=60,
                   ) -> pd.DataFrame:

    """Obter dados de preço históricos de uma ação"""
    import time

    dados_completos = []
    quantidade_por_chamada = 100
    tempo_atual = int(time.time())

    while quantidade_total > 0:
        quantidade = min(quantidade_por_chamada, quantidade_total)
        candles = api.get_candles(ativo, unidade, quantidade, tempo_atual)
        if not candles:
            break
        dados_completos.extend(candles)
        tempo_atual = candles[-1]['from'] - 1
        quantidade_total -= quantidade

    data = pd.DataFrame(dados_completos)
    data['from'] = pd.to_datetime(data['from'], unit='s')
    data["Date"] = pd.to_datetime(data["from"]).dt.date
    data.drop(["from", "id", "at", "to"], axis=1, inplace=True)

    numeric_cols = ["open", "close", "min", "max"]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    return data